# Compute climatological standard deviation

We estimate the climatological standard deviation of the state variables from a long control integration. The initial ensemble perturbations are then generated with a standard deviation equal to 5% of this climatological standard deviation, following the setup in Wilks (2005). This choice represents small but realistic analysis uncertainty relative to the system’s natural variability.

In [1]:
import sys
from pathlib import Path

# resolve project root (two levels up from this notebook)
PROJECT_ROOT = Path.cwd().resolve().parents[1]

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

for p in (SRC_DIR, NOTEBOOKS_DIR):
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

In [2]:
import numpy as np
import yaml

from models.storage import load_output_l96
from notebook_utils import generate_sweep_dict_list
from utils.config import ConfigL96, ConfigGCM
from utils.sweep_utils import get_sweep_name
from utils.loading import load_initial_states
from ensemble.storage import (
    load_output_gcm_ensemble,
)

In [3]:
# collect all data in a dictionary
data = {}
result_dir = PROJECT_ROOT / "results"

In [4]:
output_subdir = ""
# save sigma_clim values in a yaml file
output_dir = PROJECT_ROOT / "notebooks" / "evaluation" / "output" / output_subdir
output_dir.mkdir(parents=True, exist_ok=True)
sigma_clim_file = output_dir / "sigma_clim.yaml"

## Full L96 models

### Constant forcing

In [5]:
subdir = "F_20.0"
run_dir = result_dir / "l96_training_data_t20000" / subdir
config_path = run_dir / "config.yaml"
config = ConfigL96(config_path, eval_mode=True)

In [6]:
x_true, y_true, t = load_output_l96(
    config.output_dir(run_dir), backend=config.load_backend
)

In [7]:
sigma_clim = np.std(x_true)
perc = 0.05
print(f"Climatological standard deviation: {sigma_clim:.4f}")
print(f"Climatological mean: {np.mean(x_true):.4f}")
print(f"Initial condition perturbation: {perc * sigma_clim:.4f}")
data["truth"] = {"sigma_clim": sigma_clim, "mean_clim": np.mean(x_true)}

Climatological standard deviation: 5.0711
Climatological mean: 3.7716
Initial condition perturbation: 0.2536


In [8]:
for i in range(x_true.shape[1]):
    print(f"Variable {i + 1}: {np.std(x_true[:, i]):.4f}")

Variable 1: 5.0686
Variable 2: 5.0755
Variable 3: 5.0612
Variable 4: 5.0634
Variable 5: 5.0722
Variable 6: 5.0805
Variable 7: 5.0775
Variable 8: 5.0700


### Initial states standard deviation

In [9]:
result_dir = PROJECT_ROOT / "results"
init_states_dir = result_dir / "init_states_300"

sweep = init_states_dir / "sweep.yaml"
with open(sweep, "r") as f:
    sweep_dict = yaml.safe_load(f)
sweep_list = generate_sweep_dict_list(sweep_dict)

In [10]:
states_stds = {}
for s in sweep_list:
    sweep_dir = get_sweep_name(s)
    print(f"Processing sweep dir {sweep_dir}")
    x, _, t = load_initial_states(init_states_dir / sweep_dir)
    states_stds[sweep_dir] = {"std": np.std(x), "mean": np.mean(x)}


Processing sweep dir F_18.0
Processing sweep dir F_20.0


In [11]:
for k, v in states_stds.items():
    print(f"Sweep: {k}")
    print(f"  Mean: {v['mean']}")
    print(f"  Std: {v['std']}\n")

Sweep: F_18.0
  Mean: 3.658111789063055
  Std: 4.577706158366536

Sweep: F_20.0
  Mean: 3.7622584316601757
  Std: 5.067154172732835



## Reduced L96 model (parameterized)

In [12]:
gcm_dirs = [
    result_dir / "ensemble_gcm_baseline_det",
    result_dir / "ensemble_gcm_baseline_ar_p",
    result_dir / "ensemble_gcm_bayes",
    result_dir / "ensemble_gcm_flow",
    result_dir / "ensemble_gcm_ar_base_flow",
    result_dir / "ensemble_gcm_history_flow",
    result_dir / "ensemble_gcm_forcing_flow",
    result_dir / "ensemble_gcm_tail_flow",
]

In [13]:
def load_yaml(file_path):
    with open(file_path, "r") as f:
        return yaml.safe_load(f)


def parse_noise(sweep):
    noise_dict = sweep.get("__noise_bundle__")
    noise_type = noise_dict.get("noise_type")
    ar_order = noise_dict.get("ar_order")
    if noise_type == "ar_p":
        return f"ar_p-{int(ar_order)}"
    elif noise_type == "white":
        return "white"
    else:
        raise ValueError(f"Unknown noise_type: {noise_type}")


def model_sweep_to_model_order(model_name, sweep):
    if model_name.startswith("ensemble_gcm_baseline_det"):
        return "ensemble_gcm_baseline_det"
    elif model_name.startswith("ensemble_gcm_baseline_ar_p"):
        ar_order = sweep.get("ar_order", 1)
        return f"ensemble_gcm_baseline_ar_p-{int(ar_order)}"
    elif model_name.startswith("ensemble_gcm_bayes"):
        return "ensemble_gcm_bayes"
    elif model_name.startswith("ensemble_gcm_flow"):
        noise_label = parse_noise(sweep)
        return f"ensemble_gcm_flow-{noise_label}"
    elif model_name.startswith("ensemble_gcm_ar_base_flow"):
        noise_label = parse_noise(sweep)
        return f"ensemble_gcm_ar_base_flow-{noise_label}"
    elif model_name.startswith("ensemble_gcm_history_flow"):
        noise_label = parse_noise(sweep)
        delta_t = sweep.get("delta_t")
        return f"ensemble_gcm_history_flow-delta_t_{delta_t}-{noise_label}"
    elif model_name.startswith("ensemble_gcm_forcing_flow"):
        noise_label = parse_noise(sweep)
        return f"ensemble_gcm_forcing_flow-{noise_label}"
    elif model_name.startswith("ensemble_gcm_tail_flow"):
        noise_label = parse_noise(sweep)
        return f"ensemble_gcm_tail_flow-{noise_label}"
    else:
        raise ValueError(f"Unknown model name: {model_name}")


In [14]:
DEFAULT_SWEEP_KEY = "default"

for gcm_d in gcm_dirs:
    print(f"Loading data from {gcm_d.name}...")

    long_dir = gcm_d / "long"

    sweep_path = long_dir / "sweep.yaml"
    sweep = load_yaml(sweep_path)

    for s in generate_sweep_dict_list(sweep):
        sweep_name = get_sweep_name(s)
        out_path = long_dir / sweep_name

        config_path = out_path / "config.yaml"
        config = ConfigGCM(config_path)

        assert config.n_init_states == 10
        assert config.n_ens_members == 1
        assert config.n_models == 1
        assert config.init_states_type == "perfect"

        print(f"  Loading ensemble output for sweep: {sweep_name}...")

        x, _ = load_output_gcm_ensemble(
            config.output_dir(out_path), backend=config.load_backend
        )
        data[model_sweep_to_model_order(gcm_d.name, s)] = {
            "sigma_clim": np.std(x[0]),
            "mean_clim": np.mean(x[0]),
        }

print("Done.")

Loading data from ensemble_gcm_baseline_det...
  Loading ensemble output for sweep: ...
Loading data from ensemble_gcm_baseline_ar_p...
  Loading ensemble output for sweep: ...
Loading data from ensemble_gcm_bayes...
  Loading ensemble output for sweep: ...
Loading data from ensemble_gcm_flow...
  Loading ensemble output for sweep: noise_type_white-ar_order_0...
  Loading ensemble output for sweep: noise_type_ar_p-ar_order_1...
Loading data from ensemble_gcm_ar_base_flow...
  Loading ensemble output for sweep: noise_type_white-ar_order_0...
  Loading ensemble output for sweep: noise_type_ar_p-ar_order_1...
Loading data from ensemble_gcm_history_flow...
  Loading ensemble output for sweep: noise_type_white-ar_order_0-delta_t_1...
  Loading ensemble output for sweep: noise_type_white-ar_order_0-delta_t_2...
  Loading ensemble output for sweep: noise_type_ar_p-ar_order_1-delta_t_1...
  Loading ensemble output for sweep: noise_type_ar_p-ar_order_1-delta_t_2...
Loading data from ensemble_gc

In [15]:
for k, v in data.items():
    print(f"Model: {k}")
    print(f"  Climatological std: {v['sigma_clim']:.4f}")
    print(f"  Climatological mean: {v['mean_clim']:.4f}\n")

Model: truth
  Climatological std: 5.0711
  Climatological mean: 3.7716

Model: ensemble_gcm_baseline_det
  Climatological std: 4.9193
  Climatological mean: 3.8993

Model: ensemble_gcm_baseline_ar_p-1
  Climatological std: 4.9109
  Climatological mean: 3.8291

Model: ensemble_gcm_bayes
  Climatological std: 4.9213
  Climatological mean: 3.9045

Model: ensemble_gcm_flow-white
  Climatological std: 4.9722
  Climatological mean: 3.9077

Model: ensemble_gcm_flow-ar_p-1
  Climatological std: 4.9777
  Climatological mean: 3.8970

Model: ensemble_gcm_ar_base_flow-white
  Climatological std: 4.8949
  Climatological mean: 3.9166

Model: ensemble_gcm_ar_base_flow-ar_p-1
  Climatological std: 4.9417
  Climatological mean: 3.8710

Model: ensemble_gcm_history_flow-delta_t_1-white
  Climatological std: 4.9166
  Climatological mean: 3.9217

Model: ensemble_gcm_history_flow-delta_t_2-white
  Climatological std: 4.9364
  Climatological mean: 3.9162

Model: ensemble_gcm_history_flow-delta_t_1-ar_p-1
  

In [16]:
def to_py(obj):
    if isinstance(obj, dict):
        return {k: to_py(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_py(v) for v in obj]
    if isinstance(obj, np.generic):  # np.float32, np.int64, etc.
        return obj.item()
    return obj


with open(sigma_clim_file, "w") as f:
    yaml.safe_dump(to_py(data), f, sort_keys=True)